In [ ]:
# %% [markdown]
# # Lab 3: Testing and Evaluating Gemini LLM Functions
#
# This notebook demonstrates how to build, test, and evaluate two Gemini-powered
# functions using ipytest for inline unit tests and Vertex AI Evaluation API for
# systematic prompt comparison.

In [22]:
## Install required packages
#!pip install -q google-cloud-aiplatform ipytest pytest
#print("Installation complete. Restart the kernel before proceeding.")

Installation complete. Restart the kernel before proceeding.


In [1]:
# Import all required libraries

import re
import time
import pandas as pd
import ipytest
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from vertexai.evaluation import EvalTask

ipytest.autoconfig()
print("Imports complete.")

Imports complete.


In [2]:
# Configuration
# Project configuration — update PROJECT_ID before running
PROJECT_ID = "qwiklabs-gcp-00-16d0362ac1ac"   # <-- replace with your GCP project ID
LOCATION = "us-east4"
GEMINI_MODEL = "gemini-2.5-flash"

print(f"Project:  {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model:    {GEMINI_MODEL}")

Project:  qwiklabs-gcp-00-16d0362ac1ac
Location: us-east4
Model:    gemini-2.5-flash


In [3]:
# Initialize Vertex AI SDK and create GenerativeModel client
vertexai.init(project=PROJECT_ID, location=LOCATION)

model = GenerativeModel(
    model_name=GEMINI_MODEL,
    generation_config=GenerationConfig(temperature=0.1),
)

print(f"GenerativeModel initialized: {GEMINI_MODEL}")


GenerativeModel initialized: gemini-2.5-flash


In [4]:
# classify_question() Function
#
# Classifies a user question into one of four government service categories:
# - Employment
# - General Information
# - Emergency Services
# - Tax Related
#
# Two prompt variants are defined so they can be swapped for evaluation.

# ---------------------------------------------------------------------------
# Prompt variant V1: simple, minimal instruction
# ---------------------------------------------------------------------------
PROMPT_V1 = """You are a government services router.
Classify the following question into EXACTLY ONE of these categories:
- Employment
- General Information
- Emergency Services
- Tax Related

Respond with ONLY the category name, nothing else.

Question: {question}
Category:"""

# ---------------------------------------------------------------------------
# Prompt variant V2: few-shot with 2 examples per category
# ---------------------------------------------------------------------------
PROMPT_V2 = """You are a government services router. Your task is to read a
citizen's question and classify it into exactly one of the following four
categories by returning only the category label.

Categories and representative examples:

EMPLOYMENT
  Example 1: "How do I apply for unemployment benefits?"
  Example 2: "What documents do I need to file for workers' compensation?"

GENERAL INFORMATION
  Example 1: "What are the city hall hours of operation?"
  Example 2: "Where can I find the voter registration form?"

EMERGENCY SERVICES
  Example 1: "There is a gas leak on Main Street, who do I call?"
  Example 2: "My neighbor's house is on fire and I can't reach 911, help!"

TAX RELATED
  Example 1: "When is the deadline for property tax payments?"
  Example 2: "How do I appeal my income tax assessment?"

Now classify the question below. Return ONLY the category label — no
punctuation, no explanation, just one of: Employment, General Information,
Emergency Services, Tax Related.

Question: {question}
Category:"""

# ---------------------------------------------------------------------------
# Valid categories (used for response normalization)
# ---------------------------------------------------------------------------
VALID_CATEGORIES = [
    "Employment",
    "General Information",
    "Emergency Services",
    "Tax Related",
]

# Lowercase → canonical mapping for robust parsing
_CATEGORY_MAP = {cat.lower(): cat for cat in VALID_CATEGORIES}


def _normalize_category(raw: str) -> str:
    """
    Map Gemini's raw text response to one of the four canonical category
    labels.  Tries an exact lowercase match first, then a startswith check
    on each token so partial / multi-line responses are handled gracefully.

    Raises ValueError if no category can be determined.
    """
    cleaned = raw.strip().lower()

    # Exact match
    if cleaned in _CATEGORY_MAP:
        return _CATEGORY_MAP[cleaned]

    # Prefix / contains match (handles responses like "Emergency Services\n...")
    for key, canonical in _CATEGORY_MAP.items():
        if cleaned.startswith(key) or key in cleaned:
            return canonical

    raise ValueError(
        f"Could not map response to a valid category. Raw response: {raw!r}"
    )


def classify_question(question: str, prompt_template: str = PROMPT_V1) -> str:
    """
    Classify a citizen question into one of four government service categories.

    Parameters
    ----------
    question : str
        The user's question text.
    prompt_template : str
        Either PROMPT_V1 (simple) or PROMPT_V2 (few-shot). Defaults to V1.

    Returns
    -------
    str
        One of: "Employment", "General Information",
                "Emergency Services", "Tax Related".

    Raises
    ------
    ValueError
        If the question is empty or Gemini returns an unrecognisable response.
    """
    if not question or not question.strip():
        raise ValueError("question must be a non-empty string.")

    classify_model = GenerativeModel(
        model_name=GEMINI_MODEL,
        generation_config=GenerationConfig(temperature=0.1),
    )

    prompt = prompt_template.format(question=question)
    response = classify_model.generate_content(prompt)
    raw_text = response.text.strip()
    return _normalize_category(raw_text)


print("classify_question() defined with PROMPT_V1 and PROMPT_V2.")

classify_question() defined with PROMPT_V1 and PROMPT_V2.


In [5]:
# generate_social_post() Function
#
# Generates a government social media post for a given announcement.
# Two prompt variants are provided for evaluation.

# ---------------------------------------------------------------------------
# Platform character limits
# ---------------------------------------------------------------------------
_PLATFORM_LIMITS = {
    "Twitter": 280,
    "Facebook": 2000,
    "Instagram": 2200,
    "LinkedIn": 3000,
}

# ---------------------------------------------------------------------------
# Prompt variant V1: minimal instruction
# ---------------------------------------------------------------------------
SOCIAL_PROMPT_V1 = """Write a government social media post for the following
announcement. Platform: {platform}. Tone: {tone}.

Announcement: {announcement_text}

Post:"""

# ---------------------------------------------------------------------------
# Prompt variant V2: detailed with emoji guidance, character limits, and CTA
# ---------------------------------------------------------------------------
SOCIAL_PROMPT_V2 = """You are a government communications specialist writing
an official social media post.

Platform: {platform}
Character limit for this platform: {char_limit}
Tone: {tone}

Announcement: {announcement_text}

Guidelines:
1. Stay within the platform's character limit.
2. Use 1-3 relevant emojis to increase engagement (e.g., 🚨 for emergencies,
   📢 for general announcements, 💼 for employment, 💰 for taxes).
3. Include a clear call-to-action (e.g., "Visit [website]", "Call [number]",
   "Share with neighbors").
4. Keep language accessible to the general public.
5. If the tone is "official", use formal language.
   If "friendly", use conversational but professional language.
6. For Twitter: be concise, use hashtags sparingly (1-2 max).
   For Facebook/LinkedIn: you may include more detail and context.
   For Instagram: focus on visual language and community engagement.

Return ONLY the post text — no labels, no metadata.

Post:"""


def generate_social_post(
    announcement_text: str,
    platform: str = "Twitter",
    tone: str = "official",
) -> str:
    """
    Generate a government social media post for the given announcement.

    Parameters
    ----------
    announcement_text : str
        The announcement content to be promoted.
    platform : str
        Target social media platform. One of: Twitter, Facebook,
        Instagram, LinkedIn. Defaults to "Twitter".
    tone : str
        Desired tone: "official" or "friendly". Defaults to "official".

    Returns
    -------
    str
        The generated social media post text.
    """
    char_limit = _PLATFORM_LIMITS.get(platform, 500)

    social_model = GenerativeModel(
        model_name=GEMINI_MODEL,
        generation_config=GenerationConfig(temperature=0.7),
    )

    prompt = SOCIAL_PROMPT_V2.format(
        platform=platform,
        char_limit=char_limit,
        tone=tone,
        announcement_text=announcement_text,
    )
    response = social_model.generate_content(prompt)
    return response.text.strip()


print("generate_social_post() defined with SOCIAL_PROMPT_V1 and SOCIAL_PROMPT_V2.")


generate_social_post() defined with SOCIAL_PROMPT_V1 and SOCIAL_PROMPT_V2.


In [6]:
# Demonstrate both functions with representative examples

print("=" * 70)
print("DEMO: classify_question()")
print("=" * 70)

demo_questions = [
    "How do I apply for unemployment benefits?",
    "There is a gas leak on Main Street, who do I call?",
    "When is the property tax payment deadline?",
    "What are the city hall hours?",
    "I lost my job last week and need help paying rent.",
]

for q in demo_questions:
    category = classify_question(q)
    print(f"\nQ: {q}")
    print(f"   → {category}")

print()
print("=" * 70)
print("DEMO: generate_social_post()")
print("=" * 70)

demo_announcements = [
    (
        "City Hall will be closed on Monday, July 4th in observance of "
        "Independence Day. Essential services remain available.",
        "Twitter",
        "official",
    ),
    (
        "Free job fair this Saturday at the Community Center. Over 50 "
        "employers hiring now. Bring your resume!",
        "Facebook",
        "friendly",
    ),
    (
        "Water main break on Oak Avenue — expect service disruptions "
        "from 8 AM to 4 PM. Crews are on-site.",
        "Twitter",
        "official",
    ),
]

for announcement, platform, tone in demo_announcements:
    post = generate_social_post(announcement, platform=platform, tone=tone)
    print(f"\n[{platform} | {tone}]")
    print(f"Announcement: {announcement[:60]}...")
    print(f"Post: {post}")

print("\nDemo complete.")


DEMO: classify_question()

Q: How do I apply for unemployment benefits?
   → Employment

Q: There is a gas leak on Main Street, who do I call?
   → Emergency Services

Q: When is the property tax payment deadline?
   → Tax Related

Q: What are the city hall hours?
   → General Information

Q: I lost my job last week and need help paying rent.
   → Employment

DEMO: generate_social_post()

[Twitter | official]
Announcement: City Hall will be closed on Monday, July 4th in observance o...
Post: 📢🇺🇸 City Hall will be closed on Monday, July 4th, in observance of Independence Day. Essential services will remain available. For details on essential services, please visit our official website. #CityHall #IndependenceDay

[Facebook | friendly]
Announcement: Free job fair this Saturday at the Community Center. Over 50...
Post: 📢 Exciting news, community! 📢

Are you ready to find your next great opportunity? We're thrilled to announce a **FREE Job Fair** happening this **Saturday at the Community 

In [7]:
# Unit Tests with ipytest
#
# All tests run inline in the notebook using `ipytest`.

%%ipytest -v

import pytest
import time

# ---------------------------------------------------------------------------
# classify_question tests
# ---------------------------------------------------------------------------

class TestClassifyQuestion:

    def test_classify_employment(self):
        result = classify_question("How do I apply for unemployment benefits?")
        assert result == "Employment", f"Expected Employment, got {result}"

    def test_classify_emergency(self):
        result = classify_question(
            "There is a gas leak on Main Street, who do I call?"
        )
        assert result == "Emergency Services", f"Expected Emergency Services, got {result}"

    def test_classify_tax(self):
        result = classify_question(
            "When is the deadline for property tax payments?"
        )
        assert result == "Tax Related", f"Expected Tax Related, got {result}"

    def test_classify_general(self):
        result = classify_question("What are the city hall hours?")
        assert result == "General Information", (
            f"Expected General Information, got {result}"
        )

    def test_classify_returns_valid_category(self):
        """Any question must return one of the four valid categories."""
        result = classify_question("I need help with my government paperwork.")
        assert result in VALID_CATEGORIES, (
            f"Returned category {result!r} is not in VALID_CATEGORIES"
        )

    def test_classify_empty_input(self):
        """
        An empty string raises ValueError (documented behavior).
        The function validates input before calling the API.
        """
        with pytest.raises(ValueError):
            classify_question("")


# ---------------------------------------------------------------------------
# generate_social_post tests
# ---------------------------------------------------------------------------

class TestGenerateSocialPost:

    @pytest.fixture(autouse=True)
    def _rate_limit_pause(self):
        """Small pause between tests to respect API rate limits."""
        yield
        time.sleep(2)

    def test_social_post_not_empty(self):
        post = generate_social_post(
            "City Hall will be closed Monday for the holiday."
        )
        assert isinstance(post, str) and len(post) > 0, "Post must be a non-empty string"

    def test_social_post_twitter_length(self):
        """Twitter posts must be at most 280 characters."""
        post = generate_social_post(
            "Emergency road closure on Main Street due to flooding.",
            platform="Twitter",
            tone="official",
        )
        assert len(post) <= 280, (
            f"Twitter post too long: {len(post)} chars.\nPost: {post}"
        )

    def test_social_post_contains_announcement_context(self):
        """Response should reference the topic of the announcement."""
        announcement = "Free COVID-19 vaccines are available at City Clinic."
        post = generate_social_post(announcement, platform="Facebook", tone="official")
        # Check that at least one key topic word appears in the post
        keywords = {"vaccine", "vaccines", "covid", "clinic", "health", "free"}
        post_lower = post.lower()
        matched = any(kw in post_lower for kw in keywords)
        assert matched, (
            f"Post does not reference announcement topic.\nPost: {post}"
        )

    def test_social_post_different_platforms(self):
        """Twitter and Facebook posts for the same announcement should differ."""
        announcement = (
            "The city is hiring 200 new public works employees. "
            "Apply online at cityworks.gov."
        )
        twitter_post = generate_social_post(
            announcement, platform="Twitter", tone="official"
        )
        time.sleep(2)
        facebook_post = generate_social_post(
            announcement, platform="Facebook", tone="official"
        )
        assert twitter_post != facebook_post, (
            "Twitter and Facebook posts should differ for the same announcement."
        )

    def test_social_post_different_tones(self):
        """Official and friendly tone should produce different outputs."""
        announcement = "Water bill payments are due by the end of the month."
        official_post = generate_social_post(
            announcement, platform="Twitter", tone="official"
        )
        time.sleep(2)
        friendly_post = generate_social_post(
            announcement, platform="Twitter", tone="friendly"
        )
        assert official_post != friendly_post, (
            "Official and friendly tone should produce different posts."
        )

======================================= test session starts ========================================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: langsmith-0.8.5, anyio-4.13.0, typeguard-4.5.2
collected 11 items

t_18620f6decfa44f0831aa01fa5a5f9de.py ...........                                            [100%]

========================================= warnings summary =========================================
../usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290
  /usr/local/lib/python3.12/dist-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================= 11 passed, 1 warning in 92.65s (0:01:32) =============================


In [8]:
# Vertex AI Evaluation for classify_question

# ---------------------------------------------------------------------------
# Evaluation dataset — 8 questions, 2 per category
# ---------------------------------------------------------------------------
CLASSIFY_EVAL_QUESTIONS = [
    # Employment
    {
        "question": "How do I file for unemployment insurance after being laid off?",
        "reference": "Employment",
    },
    {
        "question": "What benefits am I entitled to if I was injured at work?",
        "reference": "Employment",
    },
    # General Information
    {
        "question": "Where do I go to renew my driver's license?",
        "reference": "General Information",
    },
    {
        "question": "How do I register to vote in the upcoming election?",
        "reference": "General Information",
    },
    # Emergency Services
    {
        "question": "A tree fell on a power line outside my house — who do I contact?",
        "reference": "Emergency Services",
    },
    {
        "question": "There is a chemical smell coming from a nearby factory. Help!",
        "reference": "Emergency Services",
    },
    # Tax Related
    {
        "question": "How do I apply for a homestead property tax exemption?",
        "reference": "Tax Related",
    },
    {
        "question": "What happens if I miss the deadline for filing my state income tax?",
        "reference": "Tax Related",
    },
]

# ---------------------------------------------------------------------------
# Generate responses from each prompt variant
# ---------------------------------------------------------------------------
print("Generating classify_question responses for V1 and V2...")

v1_records = []
v2_records = []

for item in CLASSIFY_EVAL_QUESTIONS:
    question = item["question"]
    reference = item["reference"]

    # V1 response
    try:
        v1_response = classify_question(question, prompt_template=PROMPT_V1)
    except Exception as e:
        v1_response = f"ERROR: {e}"
    time.sleep(1)

    # V2 response
    try:
        v2_response = classify_question(question, prompt_template=PROMPT_V2)
    except Exception as e:
        v2_response = f"ERROR: {e}"
    time.sleep(1)

    # Build prompt string (what was sent to the model)
    v1_prompt = PROMPT_V1.format(question=question)
    v2_prompt = PROMPT_V2.format(question=question)

    v1_records.append({
        "prompt": v1_prompt,
        "response": v1_response,
        "reference": reference,
        "question": question,
    })
    v2_records.append({
        "prompt": v2_prompt,
        "response": v2_response,
        "reference": reference,
        "question": question,
    })

df_v1 = pd.DataFrame(v1_records)
df_v2 = pd.DataFrame(v2_records)

print(f"\nDataset built: {len(df_v1)} rows each.\n")
print("Sample V1 responses:")
print(df_v1[["question", "response", "reference"]].to_string(index=False))

# ---------------------------------------------------------------------------
# Accuracy comparison (predicted vs reference)
# ---------------------------------------------------------------------------
def compute_accuracy(df: pd.DataFrame) -> float:
    correct = (df["response"].str.strip() == df["reference"].str.strip()).sum()
    return correct / len(df)

v1_accuracy = compute_accuracy(df_v1)
v2_accuracy = compute_accuracy(df_v2)

print(f"\nAccuracy Comparison:")
print(f"  PROMPT_V1 accuracy: {v1_accuracy:.1%}")
print(f"  PROMPT_V2 accuracy: {v2_accuracy:.1%}")

# ---------------------------------------------------------------------------
# Vertex AI EvalTask — pointwise metrics
# ---------------------------------------------------------------------------
CLASSIFY_METRICS = ["coherence", "instruction_following", "fluency"]

print("\nRunning EvalTask for PROMPT_V1 ...")
eval_task_v1 = EvalTask(
    dataset=df_v1,
    metrics=CLASSIFY_METRICS,
    experiment="lab3-classify-v1",
)
v1_result = eval_task_v1.evaluate()

print("Running EvalTask for PROMPT_V2 ...")
eval_task_v2 = EvalTask(
    dataset=df_v2,
    metrics=CLASSIFY_METRICS,
    experiment="lab3-classify-v2",
)
v2_result = eval_task_v2.evaluate()

# ---------------------------------------------------------------------------
# Side-by-side metric comparison
# ---------------------------------------------------------------------------
print("\n" + "=" * 65)
print("Classify Question — V1 vs V2 Metric Comparison")
print("=" * 65)

v1_summary = v1_result.summary_metrics
v2_summary = v2_result.summary_metrics

comparison_rows = []
for metric in CLASSIFY_METRICS:
    v1_mean_key = f"{metric}/mean"
    v2_mean_key = f"{metric}/mean"
    v1_score = v1_summary.get(v1_mean_key, float("nan"))
    v2_score = v2_summary.get(v2_mean_key, float("nan"))
    winner = "V2" if v2_score > v1_score else "V1"
    comparison_rows.append({
        "Metric": metric,
        "V1 Score": round(v1_score, 3),
        "V2 Score": round(v2_score, 3),
        "Winner": winner,
    })

df_classify_comparison = pd.DataFrame(comparison_rows)
print(df_classify_comparison.to_string(index=False))

# Add accuracy row
accuracy_row = pd.DataFrame([{
    "Metric": "accuracy (vs reference)",
    "V1 Score": round(v1_accuracy, 3),
    "V2 Score": round(v2_accuracy, 3),
    "Winner": "V2" if v2_accuracy > v1_accuracy else "V1",
}])
df_classify_comparison = pd.concat(
    [df_classify_comparison, accuracy_row], ignore_index=True
)
print("\nWith accuracy:")
print(df_classify_comparison.to_string(index=False))


Generating classify_question responses for V1 and V2...

Dataset built: 8 rows each.

Sample V1 responses:
                                                           question            response           reference
     How do I file for unemployment insurance after being laid off?          Employment          Employment
           What benefits am I entitled to if I was injured at work?          Employment          Employment
                        Where do I go to renew my driver's license? General Information General Information
                How do I register to vote in the upcoming election? General Information General Information
   A tree fell on a power line outside my house — who do I contact?  Emergency Services  Emergency Services
      There is a chemical smell coming from a nearby factory. Help!  Emergency Services  Emergency Services
             How do I apply for a homestead property tax exemption?         Tax Related         Tax Related
What happens if I miss the de

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 24 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 24/24 [00:28<00:00,  1.19s/it]
INFO:vertexai.evaluation._evaluation:All 24 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:28.596027003000927 seconds


Running EvalTask for PROMPT_V2 ...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 24 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 24/24 [02:03<00:00,  5.13s/it]
INFO:vertexai.evaluation._evaluation:All 24 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:123.16225189499892 seconds



Classify Question — V1 vs V2 Metric Comparison
               Metric  V1 Score  V2 Score Winner
            coherence       5.0       5.0     V1
instruction_following       5.0       5.0     V1
              fluency       5.0       5.0     V1

With accuracy:
                 Metric  V1 Score  V2 Score Winner
              coherence       5.0       5.0     V1
  instruction_following       5.0       5.0     V1
                fluency       5.0       5.0     V1
accuracy (vs reference)       1.0       1.0     V1


In [9]:

# ## Vertex AI Evaluation — generate_social_post
#
# Compare SOCIAL_PROMPT_V1 vs SOCIAL_PROMPT_V2 on 5 government announcements
# using pointwise metrics including verbosity.

# ---------------------------------------------------------------------------
# 5 government announcement scenarios
# ---------------------------------------------------------------------------
SOCIAL_EVAL_SCENARIOS = [
    {
        "announcement": "All public schools in the district will be closed "
                        "tomorrow, Tuesday March 5th, due to a winter storm warning.",
        "platform": "Twitter",
        "tone": "official",
    },
    {
        "announcement": "A severe thunderstorm watch is in effect for the metro area "
                        "from 6 PM to midnight. Residents are advised to stay indoors.",
        "platform": "Twitter",
        "tone": "official",
    },
    {
        "announcement": "City Hall will be closed on Monday July 4th in observance "
                        "of Independence Day. Normal operations resume Tuesday.",
        "platform": "Facebook",
        "tone": "friendly",
    },
    {
        "announcement": "Road construction on Highway 9 will begin Monday and last "
                        "12 weeks. Expect delays; detour routes are available.",
        "platform": "Facebook",
        "tone": "official",
    },
    {
        "announcement": "The county health department is urging all residents aged "
                        "65+ to get their flu vaccine this fall. Free clinics available.",
        "platform": "Twitter",
        "tone": "friendly",
    },
]

SOCIAL_METRICS = ["coherence", "fluency", "instruction_following", "verbosity"]

# ---------------------------------------------------------------------------
# Generate responses using both prompt variants
# ---------------------------------------------------------------------------
print("Generating social post responses for V1 and V2...")

social_v1_records = []
social_v2_records = []

social_model_temp = GenerativeModel(
    model_name=GEMINI_MODEL,
    generation_config=GenerationConfig(temperature=0.7),
)

for scenario in SOCIAL_EVAL_SCENARIOS:
    announcement = scenario["announcement"]
    platform = scenario["platform"]
    tone = scenario["tone"]
    char_limit = _PLATFORM_LIMITS.get(platform, 500)

    # V1 prompt (minimal)
    v1_prompt = SOCIAL_PROMPT_V1.format(
        platform=platform,
        tone=tone,
        announcement_text=announcement,
    )
    try:
        v1_resp = social_model_temp.generate_content(v1_prompt).text.strip()
    except Exception as e:
        v1_resp = f"ERROR: {e}"
    time.sleep(1)

    # V2 prompt (detailed)
    v2_prompt = SOCIAL_PROMPT_V2.format(
        platform=platform,
        char_limit=char_limit,
        tone=tone,
        announcement_text=announcement,
    )
    try:
        v2_resp = social_model_temp.generate_content(v2_prompt).text.strip()
    except Exception as e:
        v2_resp = f"ERROR: {e}"
    time.sleep(1)

    social_v1_records.append({
        "prompt": v1_prompt,
        "response": v1_resp,
        "reference": announcement,
        "platform": platform,
        "tone": tone,
    })
    social_v2_records.append({
        "prompt": v2_prompt,
        "response": v2_resp,
        "reference": announcement,
        "platform": platform,
        "tone": tone,
    })

df_social_v1 = pd.DataFrame(social_v1_records)
df_social_v2 = pd.DataFrame(social_v2_records)

print(f"Dataset built: {len(df_social_v1)} rows each.\n")
print("Sample responses:")
for i, (r1, r2) in enumerate(zip(df_social_v1["response"], df_social_v2["response"])):
    print(f"\n[{i+1}] Platform: {df_social_v1['platform'].iloc[i]} | Tone: {df_social_v1['tone'].iloc[i]}")
    print(f"  V1: {r1[:100]}...")
    print(f"  V2: {r2[:100]}...")

# ---------------------------------------------------------------------------
# Run EvalTask for both variants
# ---------------------------------------------------------------------------
print("\nRunning EvalTask for SOCIAL_PROMPT_V1 ...")
social_eval_v1 = EvalTask(
    dataset=df_social_v1,
    metrics=SOCIAL_METRICS,
    experiment="lab3-social-v1",
)
social_v1_result = social_eval_v1.evaluate()

print("Running EvalTask for SOCIAL_PROMPT_V2 ...")
social_eval_v2 = EvalTask(
    dataset=df_social_v2,
    metrics=SOCIAL_METRICS,
    experiment="lab3-social-v2",
)
social_v2_result = social_eval_v2.evaluate()

# ---------------------------------------------------------------------------
# Summary table: V1 vs V2 average scores per metric
# ---------------------------------------------------------------------------
print("\n" + "=" * 65)
print("Social Post — V1 vs V2 Average Scores per Metric")
print("=" * 65)

social_v1_summary = social_v1_result.summary_metrics
social_v2_summary = social_v2_result.summary_metrics

social_comparison_rows = []
for metric in SOCIAL_METRICS:
    mean_key = f"{metric}/mean"
    v1_score = social_v1_summary.get(mean_key, float("nan"))
    v2_score = social_v2_summary.get(mean_key, float("nan"))
    winner = "V2" if v2_score > v1_score else "V1"
    social_comparison_rows.append({
        "Metric": metric,
        "V1 Avg Score": round(v1_score, 3),
        "V2 Avg Score": round(v2_score, 3),
        "Winner": winner,
    })

df_social_comparison = pd.DataFrame(social_comparison_rows)
print(df_social_comparison.to_string(index=False))


Generating social post responses for V1 and V2...
Dataset built: 5 rows each.

Sample responses:

[1] Platform: Twitter | Tone: official
  V1: **Option 1 (Concise):**

IMPORTANT ANNOUNCEMENT: All public schools in our district will be closed t...
  V2: 🚨❄️ OFFICIAL ANNOUNCEMENT: All public schools in the district will be closed tomorrow, Tuesday, Marc...

[2] Platform: Twitter | Tone: official
  V1: **[Government Agency Name/Official City Account]**

#WeatherAlert: A Severe Thunderstorm Watch is in...
  V2: 🚨⛈️ Official advisory: A Severe Thunderstorm Watch is in effect for the metro area from 6 PM to midn...

[3] Platform: Facebook | Tone: friendly
  V1: 🇺🇸 Friendly reminder, folks! 🇺🇸

Just a heads up that City Hall will be closed on **Monday, July 4th...
  V2: Happy Friday, everyone! 📢 Just a friendly reminder that City Hall will be closed on Monday, July 4th...

[4] Platform: Facebook | Tone: official
  V1: **IMPORTANT ANNOUNCEMENT: Highway 9 Road Construction**

Please be advised 

INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:33<00:00,  1.68s/it]
INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:33.54703945299843 seconds


Running EvalTask for SOCIAL_PROMPT_V2 ...


INFO:vertexai.evaluation._evaluation:Computing metrics with a total of 20 Vertex Gen AI Evaluation Service API requests.
100%|██████████| 20/20 [00:32<00:00,  1.63s/it]
INFO:vertexai.evaluation._evaluation:All 20 metric requests are successfully computed.
INFO:vertexai.evaluation._evaluation:Evaluation Took:32.60498845099937 seconds



Social Post — V1 vs V2 Average Scores per Metric
               Metric  V1 Avg Score  V2 Avg Score Winner
            coherence           5.0           5.0     V1
              fluency           5.0           5.0     V1
instruction_following           5.0           5.0     V1
            verbosity           0.2           0.2     V1


In [10]:

# Overall Evaluation Summary
# Formatted summary of all evaluation results

print()
print("=" * 70)
print(" OVERALL EVALUATION SUMMARY — Lab 3: Gemini Function Evaluation")
print("=" * 70)
print()

# --- classify_question summary ---
print("Function: classify_question()")
print("-" * 70)
print(f"{'Metric':<30} {'V1':>10} {'V2':>10} {'Winner':>10}")
print("-" * 70)

for _, row in df_classify_comparison.iterrows():
    print(
        f"{row['Metric']:<30} {row['V1 Score']:>10.3f} {row['V2 Score']:>10.3f}"
        f" {row['Winner']:>10}"
    )

classify_v2_wins = (df_classify_comparison["Winner"] == "V2").sum()
classify_v1_wins = (df_classify_comparison["Winner"] == "V1").sum()
print()
print(
    f"  → classify_question WINNER: "
    f"{'PROMPT_V2' if classify_v2_wins >= classify_v1_wins else 'PROMPT_V1'} "
    f"(V1 wins: {classify_v1_wins}, V2 wins: {classify_v2_wins})"
)

print()
print("Function: generate_social_post()")
print("-" * 70)
print(f"{'Metric':<30} {'V1 Avg':>10} {'V2 Avg':>10} {'Winner':>10}")
print("-" * 70)

for _, row in df_social_comparison.iterrows():
    print(
        f"{row['Metric']:<30} {row['V1 Avg Score']:>10.3f} {row['V2 Avg Score']:>10.3f}"
        f" {row['Winner']:>10}"
    )

social_v2_wins = (df_social_comparison["Winner"] == "V2").sum()
social_v1_wins = (df_social_comparison["Winner"] == "V1").sum()
print()
print(
    f"  → generate_social_post WINNER: "
    f"{'SOCIAL_PROMPT_V2' if social_v2_wins >= social_v1_wins else 'SOCIAL_PROMPT_V1'} "
    f"(V1 wins: {social_v1_wins}, V2 wins: {social_v2_wins})"
)

print()
print("=" * 70)
print("Evaluation complete.  Review experiment results in Vertex AI Console.")
print("=" * 70)


 OVERALL EVALUATION SUMMARY — Lab 3: Gemini Function Evaluation

Function: classify_question()
----------------------------------------------------------------------
Metric                                 V1         V2     Winner
----------------------------------------------------------------------
coherence                           5.000      5.000         V1
instruction_following               5.000      5.000         V1
fluency                             5.000      5.000         V1
accuracy (vs reference)             1.000      1.000         V1

  → classify_question WINNER: PROMPT_V1 (V1 wins: 4, V2 wins: 0)

Function: generate_social_post()
----------------------------------------------------------------------
Metric                             V1 Avg     V2 Avg     Winner
----------------------------------------------------------------------
coherence                           5.000      5.000         V1
fluency                             5.000      5.000         V1
instruct